# Model architecture


In [ ]:
import math
import json
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
@dataclass
class ModelDims:
    n_mels: int = 80
    n_audio_ctx: int = 1500
    n_audio_state: int = 384
    n_audio_head: int = 6
    n_audio_layer: int = 4
    n_vocab: int = 51865
    n_text_ctx: int = 448
    n_text_state: int = 384
    n_text_head: int = 6
    n_text_layer: int = 4

CONFIGS = {
    "tiny": ModelDims(),
    "base": ModelDims(n_audio_state=512, n_audio_head=8, n_audio_layer=6,
                      n_text_state=512, n_text_head=8, n_text_layer=6),
    "small": ModelDims(n_audio_state=768, n_audio_head=12, n_audio_layer=12,
                       n_text_state=768, n_text_head=12, n_text_layer=12),
    "medium": ModelDims(n_audio_state=1024, n_audio_head=16, n_audio_layer=24,
                        n_text_state=1024, n_text_head=16, n_text_layer=24),
    "large": ModelDims(n_audio_state=1280, n_audio_head=20, n_audio_layer=32,
                       n_text_state=1280, n_text_head=20, n_text_layer=32),
}

In [ ]:
def sinusoids(length, channels, max_timescale=10000):
    assert channels % 2 == 0
    log_timescale_increment = math.log(max_timescale) / (channels // 2 - 1)
    inv_timescales = torch.exp(-log_timescale_increment * torch.arange(channels // 2))
    scaled_time = torch.arange(length)[:, None] * inv_timescales[None, :]
    return torch.cat([scaled_time.sin(), scaled_time.cos()], dim=1)

class MultiHeadAttention(nn.Module):
    def __init__(self, n_state, n_head):
        super().__init__()
        self.n_head = n_head
        self.query = nn.Linear(n_state, n_state)
        self.key = nn.Linear(n_state, n_state, bias=False)
        self.value = nn.Linear(n_state, n_state)
        self.out = nn.Linear(n_state, n_state)

    def forward(self, x, xa=None, mask=None, kv_cache=None):
        q = self.query(x)
        if kv_cache is None or xa is None or self.key not in kv_cache:
            k = self.key(x if xa is None else xa)
            v = self.value(x if xa is None else xa)
            if kv_cache is not None and xa is not None:
                kv_cache[self.key] = k
                kv_cache[self.value] = v
        else:
            k = kv_cache[self.key]
            v = kv_cache[self.value]
        wv, qk = self.qkv_attention(q, k, v, mask)
        return self.out(wv), qk

    def qkv_attention(self, q, k, v, mask=None):
        n_batch, n_ctx, n_state = q.shape
        scale = (n_state // self.n_head) ** -0.25
        q = q.view(*q.shape[:2], self.n_head, -1).permute(0, 2, 1, 3) * scale
        k = k.view(*k.shape[:2], self.n_head, -1).permute(0, 2, 3, 1) * scale
        v = v.view(*v.shape[:2], self.n_head, -1).permute(0, 2, 1, 3)
        qk = q @ k
        if mask is not None:
            qk = qk + mask[:n_ctx, :n_ctx]
        w = F.softmax(qk.float(), dim=-1).to(q.dtype)
        return (w @ v).permute(0, 2, 1, 3).flatten(start_dim=2), qk.detach()

In [ ]:
class ResidualAttentionBlock(nn.Module):
    def __init__(self, n_state, n_head, cross_attention=False):
        super().__init__()
        self.attn = MultiHeadAttention(n_state, n_head)
        self.attn_ln = nn.LayerNorm(n_state)
        self.cross_attn = MultiHeadAttention(n_state, n_head) if cross_attention else None
        self.cross_attn_ln = nn.LayerNorm(n_state) if cross_attention else None
        n_mlp = n_state * 4
        self.mlp = nn.Sequential(nn.Linear(n_state, n_mlp), nn.GELU(), nn.Linear(n_mlp, n_state))
        self.mlp_ln = nn.LayerNorm(n_state)

    def forward(self, x, xa=None, mask=None, kv_cache=None):
        x = x + self.attn(self.attn_ln(x), mask=mask, kv_cache=kv_cache)[0]
        if self.cross_attn is not None:
            x = x + self.cross_attn(self.cross_attn_ln(x), xa, kv_cache=kv_cache)[0]
        x = x + self.mlp(self.mlp_ln(x))
        return x

In [ ]:
class AudioEncoder(nn.Module):
    def __init__(self, dims):
        super().__init__()
        self.conv1 = nn.Conv1d(dims.n_mels, dims.n_audio_state, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(dims.n_audio_state, dims.n_audio_state, kernel_size=3, stride=2, padding=1)
        self.register_buffer("positional_embedding", sinusoids(dims.n_audio_ctx, dims.n_audio_state))
        self.blocks = nn.ModuleList(
            [ResidualAttentionBlock(dims.n_audio_state, dims.n_audio_head) for _ in range(dims.n_audio_layer)]
        )
        self.ln_post = nn.LayerNorm(dims.n_audio_state)

    def forward(self, mel):
        x = F.gelu(self.conv1(mel))
        x = F.gelu(self.conv2(x))
        x = x.permute(0, 2, 1)
        x = (x + self.positional_embedding).to(x.dtype)
        for block in self.blocks:
            x = block(x)
        return self.ln_post(x)

class TextDecoder(nn.Module):
    def __init__(self, dims):
        super().__init__()
        self.token_embedding = nn.Embedding(dims.n_vocab, dims.n_text_state)
        self.positional_embedding = nn.Parameter(torch.empty(dims.n_text_ctx, dims.n_text_state))
        self.blocks = nn.ModuleList(
            [ResidualAttentionBlock(dims.n_text_state, dims.n_text_head, cross_attention=True)
             for _ in range(dims.n_text_layer)]
        )
        self.ln = nn.LayerNorm(dims.n_text_state)
        mask = torch.empty(dims.n_text_ctx, dims.n_text_ctx).fill_(float("-inf")).triu_(1)
        self.register_buffer("mask", mask, persistent=False)

    def forward(self, tokens, audio_features, kv_cache=None):
        offset = next(iter(kv_cache.values())).shape[1] if kv_cache else 0
        x = self.token_embedding(tokens) + self.positional_embedding[offset:offset + tokens.shape[-1]]
        x = x.to(audio_features.dtype)
        for block in self.blocks:
            x = block(x, audio_features, mask=self.mask, kv_cache=kv_cache)
        x = self.ln(x)
        return x @ self.token_embedding.weight.t()

In [ ]:
class ASRModel(nn.Module):
    def __init__(self, dims):
        super().__init__()
        self.dims = dims
        self.encoder = AudioEncoder(dims)
        self.decoder = TextDecoder(dims)
        nn.init.normal_(self.decoder.positional_embedding, std=0.01)

    def forward(self, mel, tokens):
        return self.decoder(tokens, self.encoder(mel))

    @property
    def num_parameters(self):
        return sum(p.numel() for p in self.parameters())

model = ASRModel(CONFIGS["tiny"])
print(f"{model.num_parameters / 1e6:.1f}M")

In [ ]:
mel = torch.randn(2, 80, 3000)
tokens = torch.randint(0, 51865, (2, 24))
with torch.no_grad():
    logits = model(mel, tokens)
print(logits.shape)

In [ ]:
def init_scaled(model):
    for name, p in model.named_parameters():
        if p.dim() >= 2:
            fan_in = p.shape[-1]
            std = 1.0 / math.sqrt(fan_in)
            if name.endswith("out.weight") or ".mlp.2.weight" in name:
                std /= math.sqrt(2 * max(model.dims.n_audio_layer, model.dims.n_text_layer))
            nn.init.normal_(p, mean=0.0, std=std)
        elif "bias" in name:
            nn.init.zeros_(p)
    nn.init.normal_(model.decoder.token_embedding.weight, std=model.dims.n_text_state ** -0.5)
    nn.init.normal_(model.decoder.positional_embedding, std=0.01)
    return model

model = init_scaled(ASRModel(CONFIGS["tiny"]))

In [ ]:
def receptive_field_check(model):
    mel = torch.zeros(1, 80, 3000, requires_grad=True)
    feats = model.encoder(mel)
    feats[0, 750].sum().backward()
    grad_frames = mel.grad.abs().sum(dim=1)[0]
    nz = torch.nonzero(grad_frames > 0).flatten()
    return int(nz.min()), int(nz.max())

print(receptive_field_check(ASRModel(CONFIGS["tiny"])))

In [ ]:
def causality_check(model):
    torch.manual_seed(0)
    mel = torch.randn(1, 80, 3000)
    tokens = torch.randint(0, model.dims.n_vocab, (1, 12))
    with torch.no_grad():
        base = model(mel, tokens)
        tampered = tokens.clone()
        tampered[0, -1] = (tampered[0, -1] + 1) % model.dims.n_vocab
        out = model(mel, tampered)
    return bool(torch.allclose(base[0, :-1], out[0, :-1], atol=1e-4))

print(causality_check(model))

In [ ]:
def kv_cache_parity(model):
    torch.manual_seed(1)
    mel = torch.randn(1, 80, 3000)
    tokens = torch.randint(0, model.dims.n_vocab, (1, 10))
    model.eval()
    with torch.no_grad():
        audio = model.encoder(mel)
        full = model.decoder(tokens, audio)
        cache = {}
        step_logits = []
        for i in range(tokens.shape[1]):
            hooks = []
            def make_hook(module):
                def hook(mod, inp, out):
                    if mod in cache:
                        cache[mod] = torch.cat([cache[mod], out], dim=1)
                    else:
                        cache[mod] = out
                    return cache[mod]
                return hook
            for block in model.decoder.blocks:
                hooks.append(block.attn.key.register_forward_hook(make_hook(block.attn.key)))
                hooks.append(block.attn.value.register_forward_hook(make_hook(block.attn.value)))
            out = model.decoder(tokens[:, i:i + 1], audio, kv_cache=cache)
            step_logits.append(out)
            for h in hooks:
                h.remove()
        stepped = torch.cat(step_logits, dim=1)
    return float((full - stepped).abs().max())

print(kv_cache_parity(model))

In [ ]:
def parameter_report(dims):
    m = ASRModel(dims)
    rows = {}
    for name, p in m.named_parameters():
        top = name.split(".")[0]
        rows[top] = rows.get(top, 0) + p.numel()
    rows["total"] = sum(rows.values())
    return {k: f"{v/1e6:.2f}M" for k, v in rows.items()}

for size, dims in CONFIGS.items():
    print(size, parameter_report(dims))